# Aboriginal Oral Microbiome (Dental Care Project) Pipeline

Name: Gina Guzzo.<br>
Date: 9 Nov 2026<br>
Kernel: `rachis-qiime2-2026.10` (Python 3.12.15)

Workflow oral microbiome data of the Aboriginal Dental Care Project - cleaning and prep for analyses.<br>
<br>
This pipeline includes the full dataset of baseline and follow-up samples from multiple sequencing runs:<br>
<br>
`run_1_pilot` -> 2024 pilot data run (all baseline, all adult samples from 100 adults)<br>
`run_2`       -> 2026 Jul 9 sequencing run<br>
`run_3`       -> 2026 Aug 12 sequencing run<br>
`run_4`       -> 2026 Aug 13 sequencing run<br>
`run_5`       -> 2026 Nov 5 sequencing run<br>

## 1. SETUP

Some sections are run on the local machine and others on the cluster/HPC.<br>
<br>
Config file:


In [ ]:
import os
from pathlib import Path

# --- Where am I running? ---
ON_HPC = Path("/hpcfs").exists()

# --- Paths ---
if ON_HPC:
    PROJECT = Path("/hpcfs/users/a1640999/2026_aom_16s_processing") # adjust to path on cluster
    THREADS = 8
else:
    PROJECT = Path.home() / "/mnt/c/Users/a1640999/Documents/research_work/2026_aom_16s_processing"   # adjust to local path
    THREADS = 4

RAW      = PROJECT / "raw_files"
DATA     = PROJECT / "data"
VIS      = PROJECT / "vis"
METADATA = PROJECT / "metadata"
TMP      = PROJECT / "tmp_data"
LOGS     = PROJECT / "logs"
RUNS = {
    "run_1_pilot":     dict(source="run_1_pilot", trunc_f=250, trunc_r=250, threads=8),
#    "run_1_pilot_150": dict(source="run_1_pilot", trunc_f=150, trunc_r=150, threads=8),
    "run_2":           dict(source="run_2", trunc_f=150, trunc_r=150, threads=8),
    "run_3":           dict(source="run_3", trunc_f=150, trunc_r=150, threads=8),
    "run_4":           dict(source="run_4", trunc_f=150, trunc_r=150, threads=8),
    "run_5":           dict(source="run_5", trunc_f=150, trunc_r=150, threads=8),
}
RERUN = False   # Change to TRUE if wanting to re-submit jobs

# --- Make directories and move into the project ---
for run in RUNS:
    (DATA / run).mkdir(parents=True, exist_ok=True)
    (VIS / run).mkdir(parents=True, exist_ok=True)
TMP.mkdir(exist_ok=True)

os.chdir(PROJECT)
os.environ["TMPDIR"] = str(TMP)

print(f"On HPC: {ON_HPC} | Working in: {Path.cwd()}")

Load conda environment.

In [ ]:
CONDA_SH=${CONDA_SH:-/hpcfs/users/a1640999/tools/miniconda3/etc/profile.d/conda.sh}
source "$CONDA_SH"
!conda activate rachis-qiime2-2026.10

Import packages and set seed for stochastic processes (e.g., ordination, rarefaction, etc).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
np.random.seed(42) 

Record environments and versions.

In [ ]:
!qiime --version
!conda list --export > env_{"hpc" if ON_HPC else "local"}.txt

Run nbstripout in repo to reduce large content saved in notebook (saves time when pulling between machines).

In [ ]:
!nbstripout --install

## 2. DATA IMPORT AND PROCESSING

### 2.1 Data check before import

Setup variables for sequencing run storage directories.

In [ ]:
import re, os

ARCHIVE = Path(os.environ.get("ARCHIVE_ROOT", "/uofaresstor/arcpoh_shared"))

RUNS = {
    "run_1_pilot": dict(src="2024_pilot_data"),
    "run_2":       dict(src="2026_07_09_illumina_sequencing_run"),
    "run_3":       dict(src="2026_08_12_illumina_sequencing_run"),
    "run_4":       dict(src="2026_08_13_illumina_sequencing_run"),
    "run_5":       dict(src="2026_11_05_illumina_sequencing_run"),
}

Check metadata matches fastq files in directories.

In [ ]:
meta = pd.read_csv(PROJECT / "metadata/metadata_combined_seq_runs_1-5.tsv", sep="\t", index_col=0, dtype=str) 

rows = []
for run, p in RUNS.items():
    d = ARCHIVE / p["src"] / "fastq_files"
    r1 = sorted(d.glob("*_R1_*.fastq.gz")); r2 = sorted(d.glob("*_R2_*.fastq.gz"))
    ids = {re.sub(r"_S\d+_L\d+_R1_001\.fastq\.gz$", "", f.name) for f in r1}
    rows.append(dict(run=run, R1=len(r1), R2=len(r2),
                     GB=round(sum(f.stat().st_size for f in r1 + r2) / 1e9, 1),
                     not_in_metadata=len(ids - set(meta.index))))
pd.DataFrame(rows)

All files are in the metadata file.<br>
<br>
Run 1 files are 10x bigger than the other runs because they were sequenced at 2x250 bp and likely deeper sequencing.

### 2.2 Import to QIIME 2

Script for importing to qiime2 located in `scripts/import_to_qiime2.sh`<br>
Batch script for HPC job located in `scripts/job_import_to_qiime2.sh`

The archival storage is not mounted to HPC's compute nodes. The fastq files from each run were rsynced to raw_files/<run>, imported by SLURM, then the copies were removed.

In [ ]:
RERUN_IMPORT = False
if RERUN_IMPORT:
    for run in RUNS:
        src = ARCHIVE / RUNS[run]["src"] / "fastq_files"
        !mkdir -p {PROJECT}/raw_files/{run}
        !rsync -a {src}/ {PROJECT}/raw_files/{run}/*gz

Check that copying the fastq files worked.

In [ ]:
RERUN_IMPORT = False #set to True if wanting to run
if RERUN_IMPORT:
    for run in RUNS:
        !ls {RAW}/{run} | wc -l

Run batch script loop for importing to qiime2. Run 1 needs about 30-40 mins, runs 2 to 5 need about 3 mins each. Slurm script set to 45 mins to balance these out, but run 1 can be separated from the rest of the loop if queueing time is lengthy on the cluster.

In [ ]:
RERUN_IMPORT = False #set to True if wanting to run
if RERUN_IMPORT:
    for run in RUNS:
        !sbatch -n 4 --job-name=import_{run} scripts/slurm/job_import_to_qiime2.sh {run} {RAW}/{run} direct

Copy to local and open in Qiime 2 Viewer.

In [ ]:
RERUN_IMPORT = True #set to True if wanting to run
if RERUN_IMPORT:
    for run in RUNS:
        !rsync -rlvP -e ssh a1640999@phoenix-login1.adelaide.edu.au:/hpcfs/users/a1640999/2026_aom_16s_processing/vis/{run}/*qzv {VIS}/{run}

Prevent dada2 from re-running again.

In [ ]:
RERUN_IMPORT = False #set to True if wanting to run
if RERUN:
    for run, p in RUNS.items():
        tf, tr, th = p["trunc_f"], p["trunc_r"], p["threads"]
        !sbatch -n {th} scripts/slurm/job_dada2.sh {run} {tf} {tr} {th}
else:
    print("RERUN is False, skipping submission. DADA2 was run earlier (see history above).")

Inspect each imported demux table for read length and quality.

<br>
<b>Run 1 (pilot)

Forward reads<br>
![](vis/code_vis/run_1_pilot_forward_reads.PNG)


Reverse reads<br>
![](vis/code_vis/run_1_pilot_reverse_reads.PNG)


Demultiplexed sequence length summary<br>
![](vis/code_vis/run_1_pilot_demux_seq_summary.PNG)

<br>
<b>Run 2

Forward reads<br>
![](vis/code_vis/run_2_forward_reads.PNG)


Reverse reads<br>
![](vis/code_vis/run_2_reverse_reads.PNG)


Demultiplexed sequence length summary<br>
![](vis/code_vis/run_2_demux_seq_summary.PNG)

<br>
<b>Run 3

Forward reads<br>
![](vis/code_vis/run_3_forward_reads.PNG)


Reverse reads<br>
![](vis/code_vis/run_3_reverse_reads.PNG)


Demultiplexed sequence length summary<br>
![](vis/code_vis/run_3_demux_seq_summary.PNG)

<br>
<b>Run 4

Forward reads<br>
![](vis/code_vis/run_4_forward_reads.PNG)


Reverse reads<br>
![](vis/code_vis/run_4_reverse_reads.PNG)


Demultiplexed sequence length summary<br>
![](vis/code_vis/run_4_demux_seq_summary.PNG)

<br>
<b> Run 5

Forward reads<br>
![](vis/code_vis/run_5_forward_reads.PNG)


Reverse reads<br>
![](vis/code_vis/run_5_reverse_reads.PNG)


Demultiplexed sequence length summary<br>
![](vis/code_vis/run_5_demux_seq_summary.PNG)

### 2.2 Read length mismatch

The pilot data was sequenced at 2 x 250 bp, but given the newer runs are all 2 x 150 bp, the pilot data might need to be trimmed for consistency.<br>

Export representative sequences and plot lengths to compare.

In [ ]:
from collections import Counter

def read_fasta(path):
    seqs, cur = [], []
    for line in open(path):
        if line.startswith(">"):
            if cur: seqs.append("".join(cur)); cur = []
        else:
            cur.append(line.strip())
    if cur: seqs.append("".join(cur))
    return seqs

lengths = {}
starts = {}
for run in RUNS:
    out = DATA / run / "rep_seqs_export"
    !qiime tools export --input-path {DATA}/{run}/rep_seqs_dada2.qza --output-path {out}
    seqs = read_fasta(out / "dna-sequences.fasta")
    lengths[run] = [len(s) for s in seqs]
    starts[run] = Counter(s[:20] for s in seqs)
    print(run, len(seqs), "ASVs | median length", sorted(lengths[run])[len(seqs)//2],
          "| min", min(lengths[run]), "| max", max(lengths[run]))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
for run, l in lengths.items():
    ax.hist(l, bins=range(min(map(min, lengths.values())), max(map(max, lengths.values())) + 2),
            alpha=0.5, label=run, density=True)
ax.set_xlabel("ASV length (bp)"); ax.set_ylabel("Proportion of ASVs"); ax.legend()
plt.show()

Median length is 253 in every run. That’s the typical V4-sized amplicon (515F/806R).

Compare positions.

In [ ]:
for run, c in starts.items():
    print(run, c.most_common(3))

Each run shares common 20-mers (start of the 16S sequence after the 515F primer which was removed by the sequencer). The amplicon begins at the same position in all 4 runs.

One thing to check is the max amplicon length of 418 in the pilot run which is quite long. This could be a result of non-specific amplification where the reads merged into something off-target (eg chloroplast or mitochondrial sequences).<br>
Check number of long amplicons:

In [ ]:
l = lengths["run_1_pilot"]
print(sum(x > 260 for x in l), "pilot ASVs longer than 260 bp of", len(l))

This is a very small amount, so it's not concerning, they can be filtered out downstream. The next thing to check is the smaller reads, 169 bp min in runs 3 and 4.

In [ ]:
for run, l in lengths.items():
    print(run, "ASVs <240 bp:", sum(x < 240 for x in l), "of", len(l))

Also a very small amount, so good signs all around for amplicon length consistency. We can filter downstream for very small and large amplicons when the tables are merged with qiime feature-table filter-features and filter-seqs.

How many ASVs are shared between runs?

In [ ]:
sets = {run: set(read_fasta(DATA / run / "rep_seqs_export" / "dna-sequences.fasta")) for run in RUNS}
pilot = sets["run_1_pilot"]
for run, s in sets.items():
    if run != "run_1_pilot":
        print(run, "shares", len(pilot & s), "of", len(pilot), "pilot ASVs")

Some of these differences are biological whereas some are due to differences in error profiles. Just to be safe, dada2 will be re-run on the pilot data to trim to 2 x 150 bp.

Ensure threads in sbatch command match threads in job script.

In [ ]:
!sbatch -n 8 scripts/slurm/job_dada2.sh run_1_pilot 150 150 8 run_1_pilot_150

How many reads are now shared between runs?

In [ ]:
sets = {run: set(read_fasta(DATA / run / "rep_seqs_export" / "dna-sequences.fasta")) for run in RUNS}
pilot = sets["run_1_pilot_150"]
for run, s in sets.items():
    if run != "run_1_pilot_150":
        print(run, "shares", len(pilot & s), "of", len(pilot), "pilot 150 ASVs")

All shared ASVs have increased since trimming the pilot data, but the pilot data has also increased too... so it hasn't really shifted much.

Check read-weighted shared ASVs. This is calculated so reads that are more common are weighted more in the shared calculation and vice versa (instead of all reads weighted equally).

In [ ]:
import pandas as pd

def load_table(run):
    out = DATA / run / "table_export"
    !qiime tools export --input-path {DATA}/{run}/table_dada2.qza --output-path {out}
    !biom convert -i {out}/feature-table.biom -o {out}/feature-table.tsv --to-tsv
    return pd.read_csv(out / "feature-table.tsv", sep="\t", skiprows=1, index_col=0)

# ASV sequences are the feature IDs in the table (hashes), so match via the FASTA
def id_to_seq(run):
    ids, seqs, cur_id, cur = [], [], None, []
    for line in open(DATA / run / "rep_seqs_export" / "dna-sequences.fasta"):
        if line.startswith(">"):
            if cur_id: seqs.append("".join(cur)); cur = []
            cur_id = line[1:].strip(); ids.append(cur_id)
        else:
            cur.append(line.strip())
    seqs.append("".join(cur))
    return dict(zip(ids, seqs))

others = [r for r in RUNS if not r.startswith("run_1_pilot")]
other_seqs = set().union(*[set(id_to_seq(r).values()) for r in others])

for pilot in ["run_1_pilot", "run_1_pilot_150"]:
    tab = load_table(pilot)
    seqmap = id_to_seq(pilot)
    reads = tab.sum(axis=1)                      # total reads per ASV
    shared = reads[[seqmap[i] in other_seqs for i in reads.index]].sum()
    print(f"{pilot}: {100 * shared / reads.sum():.1f}% of reads are in ASVs found in other runs")

Basically, the trimming didn't really make any difference... but good to stick with 2 x 150 bp so all runs are processed the same way.

### 2.3 Filtering low read count samples

Some biological samples in sequencing runs 2-4 failed/didn't work properly. We used a 1,000 read cut-off, using the feature table after dada2.<br>
These samples will be re-sequenced. So, the old ones need to be filtered from the feature tables.

Build the list of low-read biological samples, the list should tally to 239 samples, as that was what was previously calculated by the team.

In [ ]:
import pandas as pd

meta = pd.read_csv(PROJECT / "metadata/metadata_combined_seq_runs_1-4.tsv",
                   sep="\t", index_col=0, dtype=str)
meta = meta[~meta.index.str.startswith("#")]      # drop the #q2:types row if present

frames = []
for run in ["run_2", "run_3", "run_4"]:
    out = DATA / run / "stats_export"
    !qiime tools export --input-path {DATA}/{run}/dada2_stats.qza --output-path {out}
    st = pd.read_csv(out / "stats.tsv", sep="\t", index_col=0, skiprows=[1])
    st = st[["input", "non-chimeric"]].rename(columns={"non-chimeric": "final_reads"})
    st["run"] = run
    frames.append(st)

allstats = pd.concat(frames).rename_axis("sample-id")
allstats = allstats.join(meta["sample_or_control"], how="left")

print("Samples not found in metadata:", allstats["sample_or_control"].isna().sum())
print(allstats.groupby(["run", "sample_or_control"]).size())

low = allstats[(allstats["sample_or_control"] == "sample") & (allstats["final_reads"] < 1000)]
print(low.groupby("run").size())
print("Total:", len(low))

Great, this list tallies to 239. Now, save the list of samples and filter from the dada2 feature tables.

In [ ]:
# Record what was removed
low[["run", "input", "final_reads"]].to_csv(PROJECT / "metadata/removed_low_reads.tsv", sep="\t")

# Make per-run ID files for QIIME 2, then filter
for run, grp in low.groupby("run"):
    ids = PROJECT / f"metadata/exclude_{run}.tsv"
    pd.DataFrame({"sample-id": grp.index}).to_csv(ids, sep="\t", index=False)
    !qiime feature-table filter-samples \
        --i-table {DATA}/{run}/table_dada2.qza \
        --m-metadata-file {ids} \
        --p-exclude-ids \
        --o-filtered-table {DATA}/{run}/table_dada2_filtered.qza

Check that the filtering worked.

In [ ]:
from qiime2 import Artifact

for run in ["run_2", "run_3", "run_4"]:
    orig = Artifact.load(str(DATA / run / "table_dada2.qza")).view(pd.DataFrame)
    filt = Artifact.load(str(DATA / run / "table_dada2_filtered.qza")).view(pd.DataFrame)
    removed_ids = set(low[low["run"] == run].index)
    expected = len(set(orig.index) - removed_ids)
    print(f"{run}: original {orig.shape[0]} samples | filtered {filt.shape[0]} | expected {expected} | "
          f"min reads in filtered table: {int(filt.sum(axis=1).min())}")

Filtering worked. Min reads are still 0 because controls are present. Confirm with:

In [ ]:
low_left = filt.sum(axis=1)
low_left = low_left[low_left < 1000]
print(meta.loc[low_left.index, "sample_or_control"].value_counts())

Therefore, only control samples have less than 1,000 reads after filtering.<br>
<br>
Feature table merging will come after decontamination, as contaminants will be specific to the dates they were processed and sequenced.

## 3. DECONTAMINATION

Feature tables exported to R to run `decontam` package.<br>
<br>
R analysis saved in `2026_aom_pipeline_full_dataset.RMD`  

## 4. PHYLOGENETIC TREE AND TAXONOMY